In [2]:
# Import essential libraries
import pandas as pd                  # For handling datasets
import numpy as np                   # For numerical computations
import matplotlib.pyplot as plt       # For visualization
import seaborn as sns                 # For advanced visualizations

# Machine Learning Libraries
from sklearn.model_selection import train_test_split  # Splitting dataset
from sklearn.preprocessing import StandardScaler      # Feature scaling
from sklearn.linear_model import LogisticRegression   # Logistic Regression model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # Model evaluation
import joblib                                        # Saving the trained model

# Ignore warnings for clean output
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
# Load the dataset (replace 'your_dataset.csv' with the actual file name)
df = pd.read_csv('landslide_dataset.csv')

# Display the first few rows of the dataset
print("First 5 rows of the dataset:")
display(df.head())

# Check basic info about the dataset
print("\nDataset Info:")
df.info()

# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Summary statistics of numerical features
print("\nSummary Statistics:")
display(df.describe())

First 5 rows of the dataset:


,Landslide_ID,Latitude,Longitude,Month,Week,Rainfall_1Day,Rainfall_3Days,Rainfall_7Days,Landslide
0,1,9.572,76.887,10,3,240.5,23.2,140.0,1
1,2,11.782,76.229,7,5,372.6,321.0,597.2,1
2,3,9.617,76.872,10,3,266.0,29.8,120.9,1
3,4,11.045,76.539,10,2,213.3,83.3,144.0,1
4,5,10.161,77.011,8,1,616.0,978.0,1293.6,1



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Landslide_ID    500 non-null    int64  
 1   Latitude        500 non-null    float64
 2   Longitude       500 non-null    float64
 3   Month           500 non-null    int64  
 4   Week            500 non-null    int64  
 5   Rainfall_1Day   500 non-null    float64
 6   Rainfall_3Days  500 non-null    float64
 7   Rainfall_7Days  500 non-null    float64
 8   Landslide       500 non-null    int64  
dtypes: float64(5), int64(4)
memory usage: 35.3 KB

Missing Values:
Landslide_ID      0
Latitude          0
Longitude         0
Month             0
Week              0
Rainfall_1Day     0
Rainfall_3Days    0
Rainfall_7Days    0
Landslide         0
dtype: int64

Summary Statistics:


,Landslide_ID,Latitude,Longitude,Month,Week,Rainfall_1Day,Rainfall_3Days,Rainfall_7Days,Landslide
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.0000
mean,250.500000,10.211506,76.560200,7.512000,2.738000,64.762780,67.412580,119.472060,0.1000
std,144.481833,0.921377,0.468616,2.868853,1.371539,73.531349,97.556789,143.557728,0.3003
min,1.000000,8.456000,74.882000,1.000000,1.000000,0.000000,0.000000,0.000000,0.0000
25%,125.750000,9.699000,76.332000,6.000000,1.000000,14.000000,1.350000,17.950000,0.0000
50%,250.500000,9.998000,76.733000,8.000000,3.000000,44.500000,24.000000,68.700000,0.0000
75%,375.250000,10.687000,76.921000,10.000000,4.000000,80.050000,88.300000,173.550000,0.0000
max,500.000000,12.715000,77.154000,12.000000,5.000000,616.000000,978.000000,1293.600000,1.0000


In [6]:
from sklearn.model_selection import train_test_split

# Separate Landslide cases (1) and Non-Landslide cases (0)
landslide_cases = df[df['Landslide'] == 1]
non_landslide_cases = df[df['Landslide'] == 0]

# Further split the non-landslide cases into:
# - Same locations as landslide cases (315 cases)
# - Different locations (135 cases)
same_location_cases = non_landslide_cases.iloc[:315]  # First 315 cases
different_location_cases = non_landslide_cases.iloc[315:]  # Last 135 cases

# Split each category separately using an 80-20 ratio
landslide_train, landslide_test = train_test_split(landslide_cases, test_size=0.2, random_state=42)
same_loc_train, same_loc_test = train_test_split(same_location_cases, test_size=0.2, random_state=42)
diff_loc_train, diff_loc_test = train_test_split(different_location_cases, test_size=0.2, random_state=42)

# Combine them to form final train and test sets
train_df = pd.concat([landslide_train, same_loc_train, diff_loc_train], axis=0).sample(frac=1, random_state=42)
test_df = pd.concat([landslide_test, same_loc_test, diff_loc_test], axis=0).sample(frac=1, random_state=42)

# Separate features and target variables
X_train = train_df.drop(columns=['Landslide_ID', 'Landslide'])
y_train = train_df['Landslide']
X_test = test_df.drop(columns=['Landslide_ID', 'Landslide'])
y_test = test_df['Landslide']

# Display the final dataset shapes
print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape: {X_test.shape}")
print(f"Training Labels Shape: {y_train.shape}")
print(f"Testing Labels Shape: {y_test.shape}")

# Check class distributions in training and testing sets
print("\nClass distribution in training set:\n", y_train.value_counts(normalize=True))
print("\nClass distribution in testing set:\n", y_test.value_counts(normalize=True))

Training Features Shape: (400, 7)
Testing Features Shape: (100, 7)
Training Labels Shape: (400,)
Testing Labels Shape: (100,)

Class distribution in training set:
 Landslide
0    0.9
1    0.1
Name: proportion, dtype: float64

Class distribution in testing set:
 Landslide
0    0.9
1    0.1
Name: proportion, dtype: float64


In [7]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit on the training data and transform both training and testing sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for better readability (Optional)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

# Display first few rows after scaling
print("First 5 rows of Scaled Training Data:\n", X_train_scaled.head())

First 5 rows of Scaled Training Data:
      Latitude  Longitude     Month      Week  Rainfall_1Day  Rainfall_3Days  \
355 -1.905468   0.977919  0.845845  0.180540       0.580931        0.970090   
201 -0.169434   0.176289  0.158166 -1.263777      -0.693341       -0.537810   
49  -1.873019   0.856395  1.189685 -0.541619       2.187518        1.508670   
58  -0.347905   0.820151 -0.185673  0.180540       1.724339        1.017021   
87  -0.347905   0.820151 -2.248711 -0.541619      -0.797768       -0.670440   

     Rainfall_7Days  
355        0.327311  
201       -0.370637  
49         0.569918  
58         0.542279  
87        -0.465558  


In [8]:
from sklearn.linear_model import LogisticRegression

# Initialize the Logistic Regression model
model = LogisticRegression()

# Train the model using the training dataset
model.fit(X_train_scaled, y_train)

# Display the model coefficients
print("Model Coefficients:\n", model.coef_)
print("\nModel Intercept:\n", model.intercept_)

Model Coefficients:
 [[-0.06652438 -0.16888162 -0.20382534  0.04033404  3.19481427  1.10946411
  -0.63068759]]

Model Intercept:
 [-5.45745426]


In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Predicting on test data
y_pred = model.predict(X_test_scaled)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Display results
print("Model Performance Metrics:")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", conf_matrix)

# Detailed Classification Report
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Model Performance Metrics:
Accuracy: 0.98
Precision: 1.0
Recall: 0.8
F1 Score: 0.8888888888888888

Confusion Matrix:
 [[90  0]
 [ 2  8]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99        90
           1       1.00      0.80      0.89        10

    accuracy                           0.98       100
   macro avg       0.99      0.90      0.94       100
weighted avg       0.98      0.98      0.98       100



In [10]:
import joblib  

# Save the trained model
joblib.dump(model, 'landslide_prediction_model.pkl')

print("Model saved successfully!")

Model saved successfully!
